# 04 - Ablation: Suicidal Thoughts

Compares a "Full Model" (all features) against an "Early-Risk Model" (suicidal-thoughts feature removed), to answer: how well can depression be predicted without a direct suicidal-thought indicator?

This does not modify `data/processed/` or any other notebook.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train_full = train.drop(columns=['Depression'])
y_train = train['Depression']
X_test_full = test.drop(columns=['Depression'])
y_test = test['Depression']

SUICIDAL_COL = 'Have you ever had suicidal thoughts ?'

X_train_early = X_train_full.drop(columns=[SUICIDAL_COL])
X_test_early = X_test_full.drop(columns=[SUICIDAL_COL])

print('Full feature set:', X_train_full.shape)
print('Early-risk feature set (no suicidal thoughts):', X_train_early.shape)

Full feature set: (22318, 39)
Early-risk feature set (no suicidal thoughts): (22318, 38)


## Scale each feature set separately

Full and early-risk feature sets have different columns, so each needs its own scaler fit on its own training data.

In [2]:
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

scaler_early = StandardScaler()
X_train_early_scaled = scaler_early.fit_transform(X_train_early)
X_test_early_scaled = scaler_early.transform(X_test_early)

## Train Full and Early-Risk models

In [3]:
log_reg_full = LogisticRegression(max_iter=100, random_state=42)
log_reg_full.fit(X_train_full_scaled, y_train)

rf_full = RandomForestClassifier(random_state=42)
rf_full.fit(X_train_full, y_train)

log_reg_early = LogisticRegression(max_iter=100, random_state=42)
log_reg_early.fit(X_train_early_scaled, y_train)

rf_early = RandomForestClassifier(random_state=42)
rf_early.fit(X_train_early, y_train)

print('All four models trained')

All four models trained


## Evaluate and compare

In [4]:
def evaluate(name, y_true, y_pred, y_proba):
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall': round(recall_score(y_true, y_pred), 4),
        'F1': round(f1_score(y_true, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_true, y_proba), 4),
    }

results = []
results.append(evaluate(
    'Full - Logistic Regression', y_test,
    log_reg_full.predict(X_test_full_scaled),
    log_reg_full.predict_proba(X_test_full_scaled)[:, 1]
))
results.append(evaluate(
    'Full - Random Forest', y_test,
    rf_full.predict(X_test_full),
    rf_full.predict_proba(X_test_full)[:, 1]
))
results.append(evaluate(
    'Early-Risk - Logistic Regression', y_test,
    log_reg_early.predict(X_test_early_scaled),
    log_reg_early.predict_proba(X_test_early_scaled)[:, 1]
))
results.append(evaluate(
    'Early-Risk - Random Forest', y_test,
    rf_early.predict(X_test_early),
    rf_early.predict_proba(X_test_early)[:, 1]
))

comparison = pd.DataFrame(results).set_index('Model')
comparison

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Full - Logistic Regression,0.8464,0.8563,0.8864,0.8711,0.9183
Full - Random Forest,0.8385,0.8512,0.8776,0.8642,0.9110
Early-Risk - Logistic Regression,0.7959,0.8073,0.8555,0.8307,0.8691
Early-Risk - Random Forest,0.7937,0.8077,0.8500,0.8283,0.8603


## Interpretation

The Early-Risk models drop `Have you ever had suicidal thoughts ?` entirely. Any drop in performance quantifies how much predictive power came specifically from that one feature versus the rest of the feature set combined.

Framing note: this shows suicidal thoughts is *associated with* / *predictive of* Depression in this dataset - it does not establish that suicidal thoughts *cause* depression, nor that either model is validated for clinical use.